# 07 Performance: baseline vs improved

The brief asks for profiling and a timed comparison of the original and improved
solution. Two steps in the pipeline were changed to run faster:

1. **Parsing the raw file.** Gateways re-send the same reading many times, so most
   rows carry a JSON payload already seen. The improved parser decodes each distinct
   (sensor, payload) pair once (`dedupe_payloads=True`).
2. **Validating the reconstruction methods.** Each artificial gap is filled using only
   12 hours of data either side (`window=72`) instead of the whole series.

For each change we check that the output is unchanged, then compare run times.

In [1]:
import cProfile, pstats, time
from pathlib import Path
import pandas as pd
from mma3001_project.load import parse_env_csv
from mma3001_project.reconstruction import evaluate_methods, summarise_errors

ROOT = Path("..")
RAW = ROOT / "data/raw/5EnvSensor_MayToDec2024_180kRows.csv"

def best_time(f, repeats=3):
    """Run f() several times; return the fastest wall-clock time [s] and the output."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = f()
        times.append(time.perf_counter() - t0)
    return min(times), out

## 1. Parsing: every row vs distinct payloads only

Timing uses `time.perf_counter` and keeps the fastest of three runs, which removes
noise from other programs running at the same time. `assert_frame_equal` checks
that both versions produce exactly the same table.

In [2]:
raw = pd.read_csv(RAW, dtype=str)
n_rows = len(raw)
n_distinct = len(raw.drop_duplicates(["sensorid", "jsondata"]))
print(f"{n_rows} uploaded rows, {n_distinct} distinct (sensor, payload) pairs "
      f"({100 * (1 - n_distinct / n_rows):.0f}% are repeats)")

t_base, long_base = best_time(lambda: parse_env_csv(RAW, dedupe_payloads=False))
t_fast, long_fast = best_time(lambda: parse_env_csv(RAW, dedupe_payloads=True))
pd.testing.assert_frame_equal(long_base, long_fast)
print(f"baseline {t_base:.2f} s, improved {t_fast:.2f} s, speed-up {t_base / t_fast:.1f}x; outputs identical")

179447 uploaded rows, 42847 distinct (sensor, payload) pairs (76% are repeats)
baseline 6.16 s, improved 3.50 s, speed-up 1.8x; outputs identical


## 2. Profile: where does the parser spend its time?

`cProfile` records every function call; sorting by `tottime` (time spent inside
each function itself) shows the bottlenecks.

In [3]:
prof = cProfile.Profile()
prof.enable()
parse_env_csv(RAW, dedupe_payloads=False)
prof.disable()
pstats.Stats(prof).sort_stats("tottime").print_stats(6)

         6521762 function calls (6521592 primitive calls) in 8.579 seconds

   Ordered by: internal time
   List reduced from 694 to 6 due to restriction <6>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
   179447    2.327    0.000    2.327    0.000 C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.13_3.13.3824.0_x64__qbz5n2kfra8p0\Lib\json\decoder.py:351(raw_decode)
        1    2.228    2.228    2.334    2.334 c:\Users\helen\OneDrive\Desktop\MMA3001proj\MMA3001-project\.venv\Lib\site-packages\pandas\io\parsers\c_parser_wrapper.py:226(read)
        1    1.409    1.409    8.450    8.450 C:\Users\helen\OneDrive\Desktop\MMA3001proj\MMA3001-project\src\mma3001_project\load.py:37(parse_env_csv)
        5    0.347    0.069    0.395    0.079 c:\Users\helen\OneDrive\Desktop\MMA3001proj\MMA3001-project\.venv\Lib\site-packages\pandas\core\internals\construction.py:999(convert)
        5    0.278    0.056    0.322    0.064 c:\Users\helen\OneDrive\Desktop\M

The top three are JSON decoding (`raw_decode`, called once per row), reading the
CSV (`read`), and the Python loop inside `parse_env_csv`. Reading the CSV has to
happen once regardless, but decoding and looping scale with the number of payloads
processed. Skipping the 76% repeats removes most of that work, which is why the
speed-up is about 1.8x rather than 4x.

## 3. Validation: whole series vs a 12-hour window

Filling one gap with linear interpolation only needs its two neighbours, and a
spline is barely affected by data far from the gap. Passing the whole series
(`window=len(dev)`) makes every fill cost grow with the series length; a 12-hour
window keeps it constant. Same seed, so both versions test the same gaps.

In [4]:
clean = pd.read_csv(ROOT / "data/processed/sensor869_clean.csv", index_col="time_utc", parse_dates=True)
dev = clean["co2_ppm"][clean.index < pd.Timestamp("2024-06-01", tz="Australia/Melbourne")]

t_whole, err_whole = best_time(lambda: evaluate_methods(dev, [1, 2], n_gaps=200, seed=1, window=len(dev)), repeats=1)
t_win, err_win = best_time(lambda: evaluate_methods(dev, [1, 2], n_gaps=200, seed=1, window=72))
print(f"whole series {t_whole:.2f} s, 12-h window {t_win:.2f} s, speed-up {t_whole / t_win:.1f}x")

both = pd.concat({"whole series": summarise_errors(err_whole)["RMSE"],
                  "12-h window": summarise_errors(err_win)["RMSE"]}, axis=1)
both["difference"] = both["12-h window"] - both["whole series"]
both.round(3)

whole series 1.24 s, 12-h window 0.45 s, speed-up 2.7x


whole series  12-h window  difference
method length                                       
cubic  1             12.505       12.505        -0.0
       2             14.057       14.057         0.0
linear 1             10.360       10.360        -0.0
       2             10.799       10.799         0.0
locf   1             12.332       12.332         0.0
       2             13.332       13.332         0.0
pchip  1             10.640       10.640        -0.0
       2             11.178       11.178         0.0

The RMSEs are identical to three decimals for all four methods, so the window
changes the cost, not the result. The cubic spline agreeing shows its dependence
on far-away data is negligible.

## 4. Summary

In [5]:
perf = pd.DataFrame({"baseline [s]": [t_base, t_whole], "improved [s]": [t_fast, t_win]},
                    index=["parse raw file (dedupe payloads)", "validate methods (12-h window)"])
perf["speed-up"] = perf["baseline [s]"] / perf["improved [s]"]
perf.to_csv(ROOT / "results/performance.csv")
perf.round(2)

,baseline [s],improved [s],speed-up
parse raw file (dedupe payloads),6.16,3.50,1.76
validate methods (12-h window),1.24,0.45,2.72


Both changes give the same output in less time. Parsing is the slowest step of the
whole pipeline and only needs to run once; the validation speed-up matters more
when the experiment is repeated (different seeds, gap lengths or test periods).